# Session 1: 머신러닝 소개와 탐색적 데이터 분석 (EDA)

**대상**: ML 배경 없는 DevOps 엔지니어  
**프로젝트**: 2019 2nd ML month with KaKR - 집값 예측  
**소요 시간**: 이론 30분 + 실습 60분

---

## Part 1: 이론 (30분)

### 1.1 머신러닝(ML)이란?

#### 규칙 기반 vs 데이터 기반 학습

**규칙 기반 (전통적 프로그래밍)**

개발자가 직접 규칙을 작성합니다.

```
if CPU사용률 > 90% and 지속시간 > 5분:
    알람 발생
elif 메모리사용률 > 85%:
    알람 발생
```

**데이터 기반 (머신러닝)**

과거 데이터에서 패턴을 학습하여 스스로 규칙을 만듭니다.

```
과거 장애 데이터 1000건 → 모델 학습 → 이상 탐지 규칙 자동 생성
```

#### DevOps 비유

| 구분 | 규칙 기반 (if-else) | 머신러닝 |
|------|---------------------|----------|
| 알람 설정 | CPU > 90%이면 알람 (사람이 임계값 설정) | 과거 장애 패턴을 학습하여 이상 징후 자동 감지 |
| 용량 계획 | 월 성장률 10% 가정하고 수동 계산 | 과거 트래픽 패턴을 학습하여 미래 사용량 예측 |
| 로그 분석 | 특정 에러 문자열 grep | 로그 패턴을 학습하여 비정상 로그 자동 분류 |

> **핵심**: 머신러닝은 "데이터에서 패턴을 찾아 예측하는 기술"입니다.  
> 규칙을 사람이 만드는 게 아니라, **데이터가 규칙을 만들어** 줍니다.

### 1.2 문제 유형: 분류 / 회귀 / 군집화

머신러닝 문제는 크게 세 가지로 나뉩니다.

| 유형 | 설명 | DevOps 예시 | 출력 |
|------|------|-------------|------|
| **분류 (Classification)** | 카테고리를 예측 | 이 로그는 정상/비정상? | 범주 (A, B, C) |
| **회귀 (Regression)** | 연속적인 숫자를 예측 | 내일 서버 트래픽은 몇 RPS? | 숫자 (3.14, 1000, ...) |
| **군집화 (Clustering)** | 비슷한 것끼리 그룹핑 | 비슷한 패턴의 서버들을 묶기 | 그룹 |

#### 이 프로젝트는 **회귀** 문제입니다

- **입력**: 집의 특성 (면적, 방 개수, 위치 등)
- **출력**: 집값 (연속적인 숫자)
- **목표**: 주어진 집 정보로 가격을 **정확하게 예측**하는 것

> **생각해보기**: DevOps에서 "내일 이 서버의 응답시간은 몇 ms일까?"를 예측하는 것도 회귀 문제입니다.

### 1.3 평가 지표: RMSE (Root Mean Squared Error)

모델이 얼마나 잘 예측했는지를 측정하는 방법이 필요합니다.

**RMSE = 예측값과 실제값 차이의 평균적인 크기**

$$\text{RMSE} = \sqrt{\frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2}$$

쉽게 말하면:
1. 각 예측에 대해 (실제값 - 예측값)을 구하고
2. 제곱해서 (음수 제거)
3. 평균을 낸 후
4. 루트를 씌운다

#### 직관적 이해

| 실제 집값 | 예측값 | 오차 |
|-----------|--------|------|
| 5억 | 4.8억 | 0.2억 |
| 3억 | 3.5억 | -0.5억 |
| 7억 | 6.9억 | 0.1억 |

RMSE가 **낮을수록** 예측이 정확합니다.

#### DevOps 비유

SLA 모니터링에서 "평균 응답시간 200ms 이하"라는 목표가 있듯이,  
ML에서는 "RMSE 10만 이하"같은 목표를 세우고 모델을 개선해 나갑니다.

> **핵심**: RMSE는 "평균적으로 예측이 얼마나 빗나가는가"를 알려줍니다.

### 1.4 이상치 (Outlier)

**이상치란?** 정상 범위를 벗어난 데이터 포인트입니다.

#### DevOps 비유

서버 응답시간 데이터를 수집했더니:
- 대부분: 50~200ms
- 한 건: 50,000ms (네트워크 장애로 인한 비정상 값)

이 50,000ms 데이터를 그대로 두면 평균 응답시간이 왜곡됩니다.  
마찬가지로 집값 데이터에서도 비정상적인 값이 모델 학습을 방해할 수 있습니다.

#### 이상치 처리 방법

1. **제거**: 이상치를 데이터에서 삭제 (이번 프로젝트에서 사용)
2. **변환**: 로그 변환 등으로 영향을 줄임
3. **대체**: 중앙값이나 평균값으로 교체

> **주의**: 이상치를 무조건 제거하면 안 됩니다. "왜 이상치인가?"를 먼저 이해해야 합니다.

---

## Part 2: 실습 (60분)

이제 실제 데이터를 가지고 탐색적 데이터 분석(EDA)을 진행합니다.  
EDA는 데이터를 모델에 넣기 전에 **데이터를 이해하는 과정**입니다.

> DevOps 비유: 새 시스템을 인수인계 받으면 먼저 아키텍처를 파악하듯이,  
> ML에서는 먼저 데이터의 구조와 특성을 파악합니다.

### 2.1 환경 설정 및 라이브러리 임포트

데이터 분석에 필요한 핵심 라이브러리를 불러옵니다.

| 라이브러리 | 용도 | DevOps 비유 |
|------------|------|-------------|
| `numpy` | 수학 연산 | 숫자 계산용 도구 |
| `pandas` | 데이터 처리 | 엑셀 같은 테이블 처리 도구 |
| `matplotlib` | 기본 시각화 | Grafana 같은 차트 도구 |
| `seaborn` | 고급 시각화 | 더 예쁜 Grafana 대시보드 |
| `sklearn` | ML 도구 모음 | ML용 종합 패키지 |

In [ ]:
# 데이터 분석 필수 라이브러리 임포트
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error

# 한글 깨짐 방지 (macOS 환경)
plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

# 그래프 스타일 설정
plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = (12, 8)

# 재현 가능한 결과를 위한 랜덤 시드 고정
# DevOps 비유: 동일 환경에서 동일 결과를 보장하는 것 (멱등성)
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print('라이브러리 로드 완료!')

### 2.2 RMSE 함수 정의

이론에서 배운 RMSE를 코드로 구현합니다.  
이 함수는 앞으로 모델의 성능을 측정할 때 계속 사용합니다.

In [ ]:
def rmse(y_true, y_pred):
    """RMSE (Root Mean Squared Error) 계산 함수
    
    Args:
        y_true: 실제값 (정답)
        y_pred: 예측값 (모델이 예측한 값)
    
    Returns:
        RMSE 값 (낮을수록 좋음)
    """
    return np.sqrt(mean_squared_error(y_true, y_pred))

# RMSE 함수 테스트
# 실제 집값: [5억, 3억, 7억]
# 예측 집값: [4.8억, 3.5억, 6.9억]
실제값 = [500000000, 300000000, 700000000]
예측값 = [480000000, 350000000, 690000000]

print(f'RMSE 예시: {rmse(실제값, 예측값):,.0f}원')
print('→ 평균적으로 약 3천만원 정도 오차가 난다는 뜻입니다.')

### 2.3 데이터 로드

CSV 파일에서 데이터를 읽어옵니다.

- `train.csv`: 학습용 데이터 (정답인 price가 포함)
- `test.csv`: 테스트용 데이터 (price 없음, 우리가 예측해야 함)

> DevOps 비유: train은 과거 장애 이력(원인+결과), test는 새로운 알람(원인만 있고 결과를 예측해야 함)

In [ ]:
# CSV 파일 읽기
train = pd.read_csv('../input/train.csv')
test = pd.read_csv('../input/test.csv')

print(f'학습 데이터 크기: {train.shape}')  # (행 수, 열 수)
print(f'테스트 데이터 크기: {test.shape}')

`shape`는 (행 수, 열 수)를 알려줍니다.

- **행(row)**: 각각의 집 데이터 (데이터 포인트)
- **열(column)**: 집의 특성 (feature) - 면적, 방 개수, 위치 등

> **생각해보기**: 학습 데이터와 테스트 데이터의 열 수가 다르다면, 어떤 열이 빠져있을까요?

### 2.4 데이터 미리보기

데이터가 어떻게 생겼는지 처음 5행을 확인합니다.

In [ ]:
# 학습 데이터 처음 5행 확인
train.head()

#### 주요 컬럼(열) 설명

| 컬럼명 | 설명 | 비고 |
|--------|------|------|
| `id` | 집 고유 ID | 예측에 사용하지 않음 |
| `date` | 거래 날짜 | |
| `price` | **집값 (예측 대상!)** | 우리가 예측해야 할 값 |
| `bedrooms` | 침실 수 | |
| `bathrooms` | 욕실 수 | |
| `sqft_living` | 주거 면적 (평방피트) | 1 sqft = 약 0.09 m^2 |
| `sqft_lot` | 대지 면적 | |
| `floors` | 층 수 | |
| `waterfront` | 수변(호수/강변) 여부 | 0 또는 1 |
| `view` | 조망 등급 (0~4) | |
| `condition` | 상태 등급 (1~5) | |
| `grade` | 건축 품질 등급 (1~13) | |
| `sqft_above` | 지상 면적 | |
| `sqft_basement` | 지하실 면적 | |
| `yr_built` | 건축 연도 | |
| `yr_renovated` | 리모델링 연도 | 0이면 리모델링 안 함 |
| `zipcode` | 우편번호 (지역) | |
| `lat` | 위도 | |
| `long` | 경도 | |
| `sqft_living15` | 이웃 15채 평균 주거 면적 | |
| `sqft_lot15` | 이웃 15채 평균 대지 면적 | |

### 2.5 데이터 기본 통계

`describe()`를 사용하면 각 컬럼의 통계 요약을 한눈에 볼 수 있습니다.

> DevOps 비유: 서버 메트릭 대시보드에서 평균, 최소, 최대, p50, p75를 보는 것과 같습니다.

In [ ]:
# 기본 통계 요약
# count: 데이터 수, mean: 평균, std: 표준편차
# min/25%/50%/75%/max: 최소/1사분위/중앙값/3사분위/최대
train.describe()

> **생각해보기**: `describe()` 결과에서 price의 `mean`(평균)과 `50%`(중앙값)의 차이가 크다면, 이것은 무엇을 의미할까요?  
> (힌트: 극단적으로 비싼 집들이 평균을 끌어올리고 있을 수 있습니다.)

### 2.6 결측값 (Missing Values) 확인

결측값은 비어있는 데이터입니다. 마치 서버 로그에서 일부 필드가 빈 것과 같습니다.

결측값이 있으면 모델 학습에 문제가 생길 수 있으므로, 먼저 확인합니다.

In [ ]:
# 각 컬럼별 결측값 개수 확인
missing = train.isnull().sum()
print('=== 컬럼별 결측값 수 ===')
print(missing[missing > 0] if missing.sum() > 0 else '결측값이 없습니다!')

### 2.7 데이터 타입 확인

각 컬럼이 어떤 데이터 타입인지 확인합니다.  
숫자형(int, float)과 문자형(object)을 구분하는 것이 중요합니다.

In [ ]:
# 데이터 타입 및 메모리 사용량 확인
train.info()

---

## Part 3: 시각화 실습

데이터를 눈으로 보면 숫자만으로는 알 수 없는 패턴을 발견할 수 있습니다.

> DevOps 비유: 숫자로 된 메트릭만 보는 것과 Grafana 대시보드로 보는 것의 차이입니다.

### 3.1 히스토그램: 집값(price) 분포 확인

히스토그램은 데이터가 어떤 범위에 얼마나 몰려 있는지를 보여줍니다.

> DevOps 비유: 응답시간 분포를 보는 것과 같습니다.  
> 대부분 100ms 이하인데, 가끔 1초 이상이 있다면 히스토그램에서 긴 꼬리(long tail)가 보입니다.

In [ ]:
# 집값 분포 히스토그램
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 왼쪽: 원본 분포
axes[0].hist(train['price'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('집값 (price) 분포', fontsize=14)
axes[0].set_xlabel('가격 ($)')
axes[0].set_ylabel('빈도수')

# 오른쪽: 로그 변환한 분포
axes[1].hist(np.log1p(train['price']), bins=50, color='coral', edgecolor='white')
axes[1].set_title('집값 (price) 로그 변환 분포', fontsize=14)
axes[1].set_xlabel('log(가격)')
axes[1].set_ylabel('빈도수')

plt.tight_layout()
plt.show()

**관찰 포인트:**

- 왼쪽 그래프: 대부분의 집이 저가에 몰려 있고, 오른쪽으로 긴 꼬리가 있습니다 (오른쪽 치우침)
- 오른쪽 그래프: 로그 변환하면 종 모양(정규분포)에 가까워집니다

> **왜 중요한가?** 많은 ML 모델은 데이터가 정규분포에 가까울 때 더 잘 작동합니다.  
> 로그 변환은 치우친 분포를 정규분포에 가깝게 만드는 기법입니다.

### 3.2 산점도: 주거 면적 vs 집값

산점도는 두 변수 사이의 관계를 보여줍니다.  
면적이 넓으면 집값이 비쌀까요? 확인해봅시다.

> DevOps 비유: CPU 코어 수와 처리 성능(TPS)의 관계를 산점도로 보는 것과 같습니다.

In [ ]:
# 주거 면적(sqft_living) vs 집값(price) 산점도
plt.figure(figsize=(12, 8))
plt.scatter(train['sqft_living'], train['price'], 
            alpha=0.3,  # 투명도 (겹치는 점을 구분하기 위해)
            s=10,       # 점 크기
            color='steelblue')

plt.title('주거 면적 vs 집값', fontsize=16)
plt.xlabel('주거 면적 (sqft_living)', fontsize=12)
plt.ylabel('가격 ($)', fontsize=12)
plt.show()

**관찰 포인트:**

- 전반적으로 면적이 클수록 가격이 높은 **양의 상관관계**가 보입니다
- 하지만 같은 면적이라도 가격 차이가 큽니다 (위치, 컨디션 등 다른 요인의 영향)
- 오른쪽 아래에 **이상치**가 보일 수 있습니다: 면적은 매우 큰데 가격이 낮은 경우

> **생각해보기**: 오른쪽 하단의 이상치는 어떤 집일까요? 면적이 12,000 sqft 이상인데 가격이 300만 달러 미만인 경우를 찾아보세요.

### 3.3 이상치 확인 및 제거

산점도에서 발견한 이상치를 실제로 확인하고 제거합니다.

In [ ]:
# 이상치 후보 확인: 면적 > 12000 sqft이면서 가격 < 300만 달러
outliers = train[(train['sqft_living'] > 12000) & (train['price'] < 3000000)]
print(f'이상치 후보: {len(outliers)}건')
outliers[['id', 'price', 'sqft_living', 'bedrooms', 'bathrooms', 'grade']]

In [ ]:
# 이상치 제거
print(f'제거 전 데이터 수: {len(train)}')

train_clean = train[~((train['sqft_living'] > 12000) & (train['price'] < 3000000))].reset_index(drop=True)

print(f'제거 후 데이터 수: {len(train_clean)}')
print(f'제거된 데이터 수: {len(train) - len(train_clean)}')

### 3.4 박스플롯: zipcode별 가격 분포

박스플롯(상자 수염 그림)은 데이터의 분포를 한눈에 보여줍니다.

```
    ┌─────┐
    │     │  ← 25% ~ 75% 범위 (IQR)
────┼─────┼────  ← 중앙값 (50%)
    │     │
    └─────┘
  ──┤     ├──  ← 수염 (정상 범위)
    o          ← 이상치
```

상위 10개 zipcode에 대한 가격 분포를 비교해봅시다.

In [ ]:
# 데이터가 많은 상위 10개 zipcode 선택
top_zipcodes = train_clean['zipcode'].value_counts().head(10).index

# 해당 zipcode 데이터만 필터링
zip_data = train_clean[train_clean['zipcode'].isin(top_zipcodes)]

plt.figure(figsize=(14, 8))
sns.boxplot(x='zipcode', y='price', data=zip_data, 
            order=sorted(top_zipcodes),  # zipcode 정렬
            palette='Set3')

plt.title('상위 10개 Zipcode별 집값 분포', fontsize=16)
plt.xlabel('Zipcode', fontsize=12)
plt.ylabel('가격 ($)', fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

**관찰 포인트:**

- 같은 도시 안에서도 zipcode(지역)에 따라 집값 차이가 큽니다
- 일부 지역은 가격대가 높고, 일부는 낮습니다
- 각 박스의 높이(IQR)가 다르면 해당 지역의 가격 편차가 크다는 뜻입니다

> **생각해보기**: 가장 비싼 zipcode는 어디인가요? 가격 편차가 가장 큰 지역은?

### 3.5 지도 시각화: 위도/경도 기반 가격 분포

위도(lat)와 경도(long) 정보를 활용하여 지도 위에 집값을 표시합니다.  
색상으로 가격을 표현하면 어떤 지역이 비싼지 한눈에 파악할 수 있습니다.

> DevOps 비유: CDN 노드별 응답 시간을 세계 지도 위에 색상으로 표시하는 것과 비슷합니다.

In [ ]:
# 위도/경도 기반 집값 시각화
plt.figure(figsize=(12, 10))

scatter = plt.scatter(
    train_clean['long'],      # x축: 경도
    train_clean['lat'],       # y축: 위도
    c=train_clean['price'],   # 색상: 가격
    cmap='YlOrRd',            # 색상 맵 (노랑 → 빨강)
    alpha=0.5,                # 투명도
    s=5                       # 점 크기
)

plt.colorbar(scatter, label='가격 ($)')
plt.title('위치별 집값 분포 (위도/경도)', fontsize=16)
plt.xlabel('경도 (Longitude)', fontsize=12)
plt.ylabel('위도 (Latitude)', fontsize=12)
plt.tight_layout()
plt.show()

**관찰 포인트:**

- 빨간색이 진한 곳이 집값이 비싼 지역입니다
- 특정 지역에 비싼 집들이 밀집해 있는 것을 확인할 수 있습니다
- 위치(location)가 집값에 매우 중요한 요소임을 시각적으로 알 수 있습니다

> **인사이트**: 부동산에서 "위치, 위치, 위치"라는 말이 데이터로도 확인됩니다.

### 3.6 상관관계 히트맵

각 변수들이 집값(price)과 얼마나 관련이 있는지 상관계수로 확인합니다.

- **상관계수 1.0**: 완벽한 양의 상관 (A가 오르면 B도 오름)
- **상관계수 0.0**: 관련 없음
- **상관계수 -1.0**: 완벽한 음의 상관 (A가 오르면 B가 내림)

In [ ]:
# price와의 상관계수 계산 (숫자형 컬럼만)
numeric_cols = train_clean.select_dtypes(include=[np.number]).columns
correlations = train_clean[numeric_cols].corr()['price'].sort_values(ascending=False)

print('=== price와의 상관계수 (높은 순) ===')
print(correlations)

In [ ]:
# 상관관계 히트맵 (주요 변수만)
# price와 상관계수가 높은 상위 10개 변수 선택
top_corr_cols = correlations.head(10).index.tolist()

plt.figure(figsize=(10, 8))
sns.heatmap(
    train_clean[top_corr_cols].corr(),
    annot=True,       # 숫자 표시
    fmt='.2f',        # 소수점 2자리
    cmap='RdYlBu_r',  # 색상 맵
    center=0,         # 0을 중심으로 색상 배치
    square=True       # 정사각형 셀
)
plt.title('주요 변수 간 상관관계 히트맵', fontsize=14)
plt.tight_layout()
plt.show()

**관찰 포인트:**

- `sqft_living`(주거 면적)이 가격과 가장 높은 상관관계를 보입니다
- `grade`(건축 품질), `sqft_above`(지상 면적)도 가격과 강한 양의 상관관계가 있습니다
- `sqft_living`과 `sqft_above`는 서로 상관관계가 높습니다 (다중공선성)

> **DevOps 비유**: CPU 사용률과 메모리 사용률이 함께 올라가는 것처럼,  
> 주거 면적과 지상 면적도 함께 증가하는 경향이 있습니다.

### 3.7 데이터 전처리: train/test 통합

나중에 Feature Engineering(특성 공학)을 할 때, train과 test에 동일한 처리를 적용해야 합니다.  
미리 두 데이터를 합쳐놓으면 편리합니다.

In [ ]:
# train/test 데이터 합치기
train_copy = train_clean.copy()
train_copy['data'] = 'train'  # 출처 표시

test_copy = test.copy()
test_copy['data'] = 'test'
test_copy['price'] = np.nan   # test에는 price가 없으므로 NaN 처리

# 두 데이터 합치기
data = pd.concat([train_copy, test_copy], sort=False).reset_index(drop=True)

# 불필요한 컬럼 정리
data.drop('date', axis=1, inplace=True)       # 날짜는 사용하지 않음
data['zipcode'] = data['zipcode'].astype(str)  # zipcode를 문자열로 변환

print(f'합친 데이터 크기: {data.shape}')
print(f'train: {len(data[data["data"] == "train"])}건')
print(f'test:  {len(data[data["data"] == "test"])}건')

> **왜 합치나요?**  
> 나중에 "침실 수가 3이면 1, 아니면 0"같은 변환을 할 때,  
> train에만 적용하고 test에 빼먹는 실수를 방지하기 위해서입니다.  
> `data` 컬럼으로 train/test를 구분할 수 있습니다.

---

## Session 1 요약

### 오늘 배운 것

| 주제 | 핵심 내용 |
|------|----------|
| ML이란? | 데이터에서 패턴을 찾아 예측하는 기술 |
| 문제 유형 | 이 프로젝트는 회귀 문제 (연속적인 숫자 예측) |
| 평가 지표 | RMSE - 낮을수록 좋음 |
| EDA | 데이터를 시각화하고 이해하는 과정 |
| 이상치 | 비정상 데이터를 찾아 제거 |
| 상관관계 | sqft_living, grade가 price와 가장 관련 높음 |

### 다음 Session 예고

**Session 2: Feature Engineering (특성 공학)**
- 기존 데이터에서 새로운 변수를 만들어 모델 성능을 높이는 방법
- 예: 건축 연도 → 건물 나이, zipcode → 지역 평균 가격

---

## 연습 문제

아래 문제들을 직접 코드로 풀어보세요. 각 문제 아래의 빈 코드 셀에 답을 작성하세요.

### 문제 1: 데이터 탐색

`bedrooms`(침실 수) 컬럼에 대해 다음을 확인하세요:
1. 침실 수의 최소/최대/평균값은?
2. 침실 수별 데이터 건수는? (`value_counts()` 사용)
3. 침실 수가 비정상적으로 많은 이상치가 있나요?

**힌트**: `train_clean['bedrooms'].describe()`, `train_clean['bedrooms'].value_counts()`

In [ ]:
# 여기에 문제 1의 답을 작성하세요


### 문제 2: 시각화

`grade`(건축 품질) vs `price`(가격)의 관계를 산점도로 그려보세요.

**힌트**: 위에서 `sqft_living` vs `price` 산점도를 그린 코드를 참고하세요.  
`plt.scatter(train_clean['grade'], train_clean['price'], ...)`

In [ ]:
# 여기에 문제 2의 답을 작성하세요


### 문제 3: 조건 필터링

다음 조건에 해당하는 집의 평균 가격을 각각 구해보세요:
1. `waterfront == 1` (수변 주택)
2. `waterfront == 0` (일반 주택)

수변 주택이 얼마나 더 비싼가요?

**힌트**: `train_clean[train_clean['waterfront'] == 1]['price'].mean()`

In [ ]:
# 여기에 문제 3의 답을 작성하세요


### 문제 4 (도전): 커스텀 시각화

`yr_built`(건축 연도)별 평균 집값의 변화 추이를 **선 그래프**로 그려보세요.

**힌트**:
1. `groupby`로 건축 연도별 평균 가격 계산: `train_clean.groupby('yr_built')['price'].mean()`
2. `plt.plot()`으로 선 그래프 그리기

In [ ]:
# 여기에 문제 4의 답을 작성하세요
